In [3]:
from functools import reduce
from itertools import repeat
from typing import Iterable
from collections import deque

import numpy as np
from jax import numpy as jnp
from jax import Array

In [120]:
def compose_comhprension(
    operators: Iterable[Array],
    inds: Iterable[int],
    dims: Iterable[int],
) -> Array:
    ops = [jnp.identity(dim) for dim in dims]

    for op, ind in zip(operators, inds):
        ops[ind] = op

    return reduce(jnp.kron, ops)


def compose_loops(
    operators: Iterable[Array],
    inds: Iterable[int],
    qubit_dims: Iterable[int],
) -> Array:
    qubit_ops = deque()

    for ind, dim in enumerate(qubit_dims):
        if ind in inds:
            idx = inds.index(ind)
            op = operators[idx]
        else:
            op = jnp.identity(dim)
        
        qubit_ops.append(op)  

    return reduce(jnp.kron, qubit_ops)


### Test for expanding 1 operator into Hilbert space of two qubits

In [121]:
dim = 4
num_qubits = 2

dims = list(repeat(dim, num_qubits))

seed = 42
rng = np.random.default_rng(seed)

op = rng.random(size=(dim, dim))

In [122]:
%%timeit
res = compose_comhprension([op], [0], dims)

90.8 µs ± 1.26 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [123]:
%%timeit
res = compose_loops([op], [0], dims)

49 µs ± 1.5 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Test for expanding 1 operator into the Hilbert space of 5 qubits

In [124]:
dim = 4
num_qubits = 5

dims = list(repeat(dim, num_qubits))

seed = 42
rng = np.random.default_rng(seed)

op = rng.random(size=(dim, dim))

In [125]:
%%timeit
res = compose_comhprension([op], [0], dims)

438 µs ± 15.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [126]:
%%timeit
res = compose_loops([op], [0], dims)

392 µs ± 2.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Test for expanding 2 operators into the Hilbert space of 5 qubits

In [127]:
dim = 4
num_qubits = 5

dims = list(repeat(dim, num_qubits))

seed = 42
rng = np.random.default_rng(seed)

ops = rng.random(size=(2, dim, dim))
inds = (2, 3)

In [128]:
%%timeit
res = compose_comhprension(ops, inds, dims)

430 µs ± 1.69 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [129]:
%%timeit
res = compose_loops(ops, inds, dims)

394 µs ± 54.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Quick check that the two methods produce the same results

In [130]:
res_comprehension = compose_comhprension(ops, inds, dims)
res_loops = compose_loops(ops, inds, dims)
np.allclose(res_comprehension, res_loops)

True